# 02 — Extração das direções candidatas

Agora procuramos onde a representação de pedidos nocivos se separa da representação de pedidos inofensivos. Em cada camada e posição do fim da instrução, calculamos a média nociva menos a média inofensiva.

Cada diferença é apenas uma candidata. Nesta etapa ainda não afirmamos que ela causa recusa; apenas registramos um eixo estatístico que será testado por intervenção no próximo notebook.

In [ ]:
# Reusamos exatamente a configuração da linha de base para não trocar a amostra no meio do experimento.
from pathlib import Path
import json

from refusal_direction_replication.pipeline import run_extract_stage

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
config = json.loads((ROOT / "configs" / "gemma_2b.json").read_text(encoding="utf-8"))
config

## O que será calculado

Para um par concreto — um pedido nocivo e outro inofensivo — cada camada produz um vetor residual em várias posições do sufixo de conversa do Gemma. Agregamos muitos pares e calculamos:

\[
d_{l,p}=\operatorname{média}(h^{nocivo}_{l,p})-\operatorname{média}(h^{inofensivo}_{l,p}).
\]

A camada `l` diz onde o vetor foi lido; a posição `p` indica qual token do fim da instrução foi usado. A norma de cada vetor ajuda a inspecionar a escala, mas não basta para escolher o mecanismo.

In [ ]:
# Capturamos os residuais, calculamos todas as diferenças de médias e salvamos os vetores fora do Git.
extraction = run_extract_stage(ROOT)
extraction_results = extraction["results"]
extraction_results

In [ ]:
# Inspecionamos posição, quantidade e normas sem confundir vetor grande com vetor causalmente importante.
summary = {
    "position_offsets": extraction_results["position_offsets"],
    "candidate_count": extraction_results["candidate_count"],
    "first_candidate_norms": extraction_results["candidate_norms"][:12],
    "activation_shape_harmful": extraction_results["activation_shape_harmful"],
}
summary

## Leitura do resultado

O tensor tem os eixos `exemplo × camada × posição × dimensão residual`. A subtração das médias elimina o eixo dos exemplos e produz uma candidata por par camada–posição.

No artigo, essa grade é apenas o conjunto de hipóteses. O artefato `candidates.pt` fica fora do Git e será submetido aos testes de ablação, adição e KL. Diferenças de amostra, revisão do Gemma e precisão numérica podem alterar as normas sem invalidar o procedimento.